# Corte 2 — Limpieza y consulta de datos: ventas de una tienda
**Autor:** NOMBRE COMPLETO · **GitHub:** USUARIO

Dataset: `ventas_raw.csv` (pedidos de una tienda, ene–sep 2026). Contiene nulos, duplicados, tipos mal formados y formatos inconsistentes.

In [1]:
import pandas as pd, numpy as np, sqlite3
pd.set_option('display.width',140)
df_raw = pd.read_csv('ventas_raw.csv')
df = df_raw.copy()
df.head()

,id_pedido,fecha,cliente_id,cliente_nombre,ciudad,producto,categoria,precio,cantidad,metodo_pago
0,389,2026-08-25,3,Maria Gomez,NaN,Mouse,Tecnologia,45000,1.0,Efectivo
1,499,2026-09-08,3,Maria Gomez,Medellin,Audifonos,Tecnologia,NaN,4.0,NaN
2,139,2026-09-09,8,Andres Vega,Neiva,Lampara,NaN,"$85,000",5.0,Tarjeta
3,427,2026-05-21,10,Diego Castro,Cali,Cafetera,Hogar,260000,2.0,Efectivo
4,66,2026-01-11,3,NaN,Medellin,Teclado,Tecnologia,120000,1.0,Efectivo


## 1. Diagnóstico inicial (ANTES)

In [2]:
print('Filas, columnas:', df_raw.shape)
print('\nTipos:'); print(df_raw.dtypes)
print('\nNulos por columna:'); print(df_raw.isna().sum())
print('\nDuplicados exactos:', df_raw.duplicated().sum())
print('Cantidades <= 0:', (pd.to_numeric(df_raw['cantidad'],errors='coerce')<=0).sum())
print('Valores distintos de ciudad:', sorted(df_raw['ciudad'].dropna().unique()))
print('Valores distintos de producto:', sorted(df_raw['producto'].dropna().unique())[:12])

Filas, columnas: (530, 10)

Tipos:
id_pedido           int64
fecha                 str
cliente_id          int64
cliente_nombre        str
ciudad                str
producto              str
categoria             str
precio                str
cantidad          float64
metodo_pago           str
dtype: object

Nulos por columna:
id_pedido          0
fecha              0
cliente_id         0
cliente_nombre    21
ciudad            43
producto           0
categoria         31
precio            26
cantidad          26
metodo_pago       39
dtype: int64

Duplicados exactos: 30
Cantidades <= 0: 6
Valores distintos de ciudad: ['BOGOTA', 'Bogota', 'CALI', 'Cali', 'MEDELLIN', 'Medellin', 'NEIVA', 'Neiva', 'PITALITO', 'Pitalito']
Valores distintos de producto: [' audifonos  ', ' cafetera  ', ' camiseta  ', ' jean  ', ' lampara  ', ' laptop  ', ' licuadora  ', ' mouse  ', ' teclado  ', 'Audifonos', 'Cafetera', 'Camiseta']


## 2. Limpieza
### 2.1 Duplicados
Se eliminan las filas completamente repetidas.

In [3]:
df = df.drop_duplicates().reset_index(drop=True)
print('Filas tras quitar duplicados:', len(df))

Filas tras quitar duplicados: 500


### 2.2 Normalizar formatos de texto
Se quitan espacios y se unifica mayúsculas/minúsculas en `producto`, `ciudad`, `metodo_pago`, `categoria` y `cliente_nombre`.

In [4]:
for col in ['producto','ciudad','metodo_pago','categoria','cliente_nombre']:
    df[col] = df[col].str.strip().str.title()
print(sorted(df['ciudad'].dropna().unique()))
print(sorted(df['producto'].dropna().unique()))

['Bogota', 'Cali', 'Medellin', 'Neiva', 'Pitalito']
['Audifonos', 'Cafetera', 'Camiseta', 'Jean', 'Lampara', 'Laptop', 'Licuadora', 'Mouse', 'Teclado', 'Zapatos']


### 2.3 Corregir tipos
- `precio`: venía como texto con `$` y comas → numérico.
- `fecha`: mezcla de `YYYY-MM-DD` y `DD/MM/YYYY` → `datetime`.
- `cantidad`: valores `-1` son inválidos → se tratan como nulos.

In [5]:
df['precio'] = pd.to_numeric(df['precio'].astype(str).str.replace(r'[\$,]','',regex=True), errors='coerce')
f1 = pd.to_datetime(df['fecha'], format='%Y-%m-%d', errors='coerce')
f2 = pd.to_datetime(df['fecha'], format='%d/%m/%Y', errors='coerce')
df['fecha'] = f1.fillna(f2)
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')
df.loc[df['cantidad']<=0,'cantidad'] = np.nan
print(df.dtypes); print('Fechas sin parsear:', df['fecha'].isna().sum())

id_pedido                  int64
fecha             datetime64[us]
cliente_id                 int64
cliente_nombre               str
ciudad                       str
producto                     str
categoria                    str
precio                   float64
cantidad                 float64
metodo_pago                  str
dtype: object
Fechas sin parsear: 0


### 2.4 Nulos
Estrategia:
- `precio` y `categoria`: se imputan con el valor más frecuente de ese `producto` (cada producto tiene un único precio y categoría).
- `ciudad` y `cliente_nombre`: se imputan con el valor más frecuente de ese `cliente_id`.
- `cantidad`: mediana (variable numérica discreta, robusta a atípicos).
- `metodo_pago`: no se puede inferir → categoría `Desconocido`.

In [6]:
def moda_por(df, clave, col):
    m = df.dropna(subset=[col]).groupby(clave)[col].agg(lambda s: s.mode().iloc[0])
    df[col] = df[col].fillna(df[clave].map(m))
moda_por(df,'producto','precio'); moda_por(df,'producto','categoria')
moda_por(df,'cliente_id','ciudad'); moda_por(df,'cliente_id','cliente_nombre')
df['cantidad'] = df['cantidad'].fillna(df['cantidad'].median()).astype(int)
df['metodo_pago'] = df['metodo_pago'].fillna('Desconocido')
df['total'] = df['precio']*df['cantidad']
print(df.isna().sum())

id_pedido         0
fecha             0
cliente_id        0
cliente_nombre    0
ciudad            0
producto          0
categoria         0
precio            0
cantidad          0
metodo_pago       0
total             0
dtype: int64


## 3. Resumen ANTES / DESPUÉS

In [7]:
antes = {'Filas':len(df_raw),'Nulos totales':int(df_raw.isna().sum().sum()),'Duplicados':int(df_raw.duplicated().sum()),
  'Precios como texto':int(df_raw['precio'].astype(str).str.contains(r'\$').sum()),
  'Cantidades inválidas (<=0)':int((pd.to_numeric(df_raw['cantidad'],errors='coerce')<=0).sum()),
  'Ciudades distintas':df_raw['ciudad'].nunique(),'Productos distintos':df_raw['producto'].nunique()}
despues = {'Filas':len(df),'Nulos totales':int(df.isna().sum().sum()),'Duplicados':int(df.duplicated().sum()),
  'Precios como texto':0,'Cantidades inválidas (<=0)':int((df['cantidad']<=0).sum()),
  'Ciudades distintas':df['ciudad'].nunique(),'Productos distintos':df['producto'].nunique()}
resumen = pd.DataFrame({'Antes':antes,'Después':despues}); resumen

,Antes,Después
Filas,530,500
Nulos totales,186,0
Duplicados,30,0
Precios como texto,62,0
Cantidades inválidas (<=0),6,0
Ciudades distintas,10,5
Productos distintos,19,10


In [8]:
df.to_csv('ventas_clean.csv', index=False)
df.head()

,id_pedido,fecha,cliente_id,cliente_nombre,ciudad,producto,categoria,precio,cantidad,metodo_pago,total
0,389,2026-08-25,3,Maria Gomez,Medellin,Mouse,Tecnologia,45000.0,1,Efectivo,45000.0
1,499,2026-09-08,3,Maria Gomez,Medellin,Audifonos,Tecnologia,180000.0,4,Desconocido,720000.0
2,139,2026-09-09,8,Andres Vega,Neiva,Lampara,Hogar,85000.0,5,Tarjeta,425000.0
3,427,2026-05-21,10,Diego Castro,Cali,Cafetera,Hogar,260000.0,2,Efectivo,520000.0
4,66,2026-01-11,3,Maria Gomez,Medellin,Teclado,Tecnologia,120000.0,1,Efectivo,120000.0


## 4. Consultas y hallazgos
### Pregunta 1 (pandas): ¿Qué categoría genera más ingresos entre junio y septiembre?
Filtro: fecha ≥ 2026-06-01. Agregación: suma de `total` por `categoria`.

In [9]:
q1 = (df[df['fecha']>='2026-06-01'].groupby('categoria')['total'].sum().sort_values(ascending=False))
q1.map('${:,.0f}'.format)

categoria
Tecnologia    $200,100,000
Hogar          $33,280,000
Ropa           $25,730,000
Name: total, dtype: str

In [10]:
print(f"Categoría líder: {q1.index[0]} con {q1.iloc[0]/q1.sum():.1%} de los ingresos del periodo.")

Categoría líder: Tecnologia con 77.2% de los ingresos del periodo.


### Pregunta 2 (SQL): ¿Qué ciudad tiene el ticket promedio más alto entre los pagos con tarjeta?
Se carga el dataset limpio en SQLite. Filtro: `metodo_pago = 'Tarjeta'`. Agregación: `AVG(total)` por ciudad.

In [11]:
con = sqlite3.connect(':memory:')
df.to_sql('ventas', con, index=False)
q2 = pd.read_sql("""SELECT ciudad, COUNT(*) AS pedidos, ROUND(AVG(total)) AS ticket_promedio
                     FROM ventas WHERE metodo_pago = 'Tarjeta'
                     GROUP BY ciudad ORDER BY ticket_promedio DESC""", con)
q2

,ciudad,pedidos,ticket_promedio
0,Cali,30,1995500.0
1,Neiva,48,1397396.0
2,Pitalito,12,1345417.0
3,Medellin,25,792000.0
4,Bogota,28,572679.0


In [12]:
top = q2.iloc[0]
print(f"Mayor ticket promedio con tarjeta: {top['ciudad']} (${top['ticket_promedio']:,.0f} en {top['pedidos']} pedidos).")

Mayor ticket promedio con tarjeta: Cali ($1,995,500 en 30 pedidos).


**Hallazgos:**
- **P1:** Tecnología lidera con ~77% de los ingresos entre junio y septiembre ($200,1 M), muy por encima de Hogar ($33,3 M) y Ropa ($25,7 M). Esto se explica por el alto precio unitario de productos como laptops.
- **P2:** Con pago en tarjeta, Cali tiene el ticket promedio más alto (~$1.995.500 en 30 pedidos), seguida de Neiva y Pitalito; Bogotá es la más baja (~$572.700). La diferencia sugiere que en Cali se compran más artículos de alto valor con tarjeta.